# 05 - Landmark hata yayılımı
Kontrollü landmark bozunumu ile açı belirsizliğini ve kamera görüş açısına duyarlılığı inceler.

In [ ]:
import sys, zipfile
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    artifacts_root = Path('/content/drive/MyDrive/PhysioLoad_AI_Project')
    code_root = Path('/content/physioload-ai')
    if not (code_root / 'physioload_env.py').exists():
        code_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(artifacts_root / 'code' / 'physioload-source.zip') as package: package.extractall(code_root)
else:
    artifacts_root = None
    here = Path.cwd().resolve()
    code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))
from physioload_env import setup_environment
env = setup_environment(root=code_root, artifacts_root=artifacts_root)
print(env.summary())

In [ ]:
import pandas as pd
from physioload.geometry import monte_carlo_angle_uncertainty

points = {'hip': (100, 80), 'knee': (110, 150), 'ankle': (120, 220)}
rows = []
for sigma in [0.5, 1.0, 2.0, 4.0, 8.0]:
    result = monte_carlo_angle_uncertainty(points, ('hip','knee','ankle'), landmark_sigma_px=sigma, samples=1000, seed=42)
    rows.append({'sigma_px': sigma, **result})
uncertainty_table = pd.DataFrame(rows)
uncertainty_table

In [ ]:
output_path = env.metrics_dir / 'landmark_uncertainty_demo.csv'
uncertainty_table.to_csv(output_path, index=False)
print('saved:', output_path)